# Region analysis, matrices, overlap detection, and graph workflows

Build synthetic 3D label volumes, extract region properties with `RegionAnalyzer`, compute distances and overlap metrics, then build and query containment graphs with the graph API.

In [ ]:
import numpy as np
import pandas as pd
import stackview

from vistiq.utils import ArrayIteratorConfig
from vistiq.matrix.types import FULL, LOWER, UPPER, LOWER_ND, UPPER_ND, OFF_DIAGONAL, matrix_to_numpy
from vistiq.segment.analysis import RegionAnalyzer, RegionAnalyzerConfig, region_to_numpy, dataframe_to_numpy
from vistiq.segment.select import RegionFilter, RegionFilterConfig, MinFilterConfig, RangeFilterConfig
from vistiq.segment.select import ValueFilter, ValueFilterConfig, TopKFilter, TopKFilterConfig
from vistiq.analysis import DistanceCalculator, DistanceCalculatorConfig
from vistiq.matrix import MatrixAggregator, MatrixAggregatorConfig, MatrixFormatter, MatrixFormatterConfig
from vistiq.analysis.overlap import (
    LabelIntersectionCalculatorConfig,
    LabelOverlapCalculatorConfig,
    OverlapCalculator,
    metrics_calculator_configs,
    region_map_from_dataframe,
)
from vistiq.graph import (
    GraphBuilder, GraphBuilderConfig, GraphQuery, GraphQueryConfig, GraphFilter, GraphFilterConfig, GraphLike, NXGraph
)


# Synthetic volumes

Two `(10, 200, 200)` uint64 volumes (Z×Y×X):

- **`labels`** — five non-overlapping objects with integer labels `1`–`5`
- **`areas`** — two coarse regions with labels `6` and `8` (partial overlap with objects is possible; used later for coincidence / containment)

In [ ]:
labels = np.zeros((10, 200, 200), dtype=np.uint64)

# Object 1 — upper-left
labels[2:7, 20:38, 12:30] = 1

# Object 2 — center
labels[3:9, 40:70, 38:62] = 2

# Object 3 — lower-right
labels[1:6, 72:92, 68:88] = 3

# Object 4 — lower-right
labels[4:6, 112:132, 125:163] = 4

# Object 5 — lower-right
labels[7:9, 145:180, 12:88] = 5

unique_labels = np.unique(labels)
print(f"labels.shape={labels.shape}, dtype={labels.dtype}")
print(f"unique labels: {unique_labels}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((labels == l).sum())}' for l in unique_labels if l)}}}")

In [ ]:
areas = np.zeros((10, 200, 200), dtype=np.uint64)

# Area 1 — upper-left
areas[2:9, 10:58, 10:98] = 6

# Area 2 — lower-right
areas[1:10, 96:192, 58:178] = 8

unique_areas = np.unique(areas)
print(f"areas.shape={areas.shape}, dtype={areas.dtype}")
print(f"unique labels: {unique_areas}")
print(f"voxel counts: {{{', '.join(f'{int(l)}: {int((areas == l).sum())}' for l in unique_areas if l)}}}")

In [ ]:
stackview.side_by_side(labels, areas)

# Identification and Analysis of Regions (as individual objects)

## The RegionAnalyzer (dataframe output)

Analyze the full 3D volume (`slice_def=()`). With `map_axes=True`, vector properties such as `cross_sectional_area` and `aspect_ratio` are expanded to plane-specific columns (`-xy`, `-xz`, `-yz`) based on the order of axes labels in the provided a_metadata and l_metadata annotations.

The RegionAnalyzer itself is stateless and the same RegionAnalyzerConfig can be reused for multiple analysis runs. 

Each region is assigned a unique `object_id`. Similarly each slice (dims defined by iterator_config) and stack are assigned unique `slice_id` and `stack_id` identifiers. 

**Notes:**
- Rerunning the RegionAnalyzer on the same labeled array will result in new `object_id`, `slice_id` and `stack_id`.
- bbox coordinates reflect array index positions; centroid coordinates, volume, area measurements, etc are scaled to real world dimension if `scale` is provided in the metadata. 



In [ ]:
a_metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
    "channel_names": ["Area"],
}
l_metadata = {
    "axes": ["Z", "Y", "X"],
    "scale": (2.0, 1.0, 1.0),
    "channel_names": ["Label"],
}

config = RegionAnalyzerConfig(
    output_type="dataframe",
    map_axes=True,
    index_on = "object_id",
    properties=[
        "label",
        "volume",
        "centroid",
        "bbox",
        "aspect_ratio",
        "cross_sectional_area",
    ],
    iterator_config=ArrayIteratorConfig(slice_def=()),
)

l_regions = RegionAnalyzer(config).run(labels, metadata=l_metadata)
a_regions = RegionAnalyzer(config).run(areas, metadata=a_metadata)

In [ ]:
l_regions

In [ ]:
a_regions

## Filter Regions

A RegionFilter is defined by a list of individual numeric filters that can be applied to specific region attributes. At the moment the list of filters is evaluated as a concatenation of boolean AND operations.

The following RegionFilter returns a subset of regions whwere `4000 < volume < 300000` and `aspect_ratio > 0.015`.

In [ ]:
rfcfg = RegionFilterConfig(
    filters=[
        RangeFilterConfig(
            attribute="volume",
            range=(4000.0,300000)
        ),
        MinFilterConfig(
            attribute="aspect_ratio",
            minimum=0.015,
        ),
    ]
)
l_accepted, _ = RegionFilter(rfcfg).run(l_regions)
a_accepted, _ = RegionFilter(rfcfg).run(a_regions)

In [ ]:
l_accepted

In [ ]:
a_accepted

# Analysis of Inter-Region Metrics 

The RegionAnalyzer provides rich metrics for each identified region/object individually. In addition we can extract spatial (or temporal) relationships between identified regions. 

For example:
- inter-object distance
- k-nearest neighbors 
- r-nearest neighbors (number of neighbors within certain radius)
- overlap of regions and containment

These calculations can be mapped to matrix operations. The core tools:
- MatrixBuilder
    - LabelBuilder:  build matrix representation for two n-dimentsional labeled image stacks (e.g. from MicroSAMSegmenter output)
    - MaskBuilder: build matrix representation for two sets of binary n-dimensional masks
    - BBoxBuilder: build matrix representation for two sets of bounding boxes (e.g. from RegionAnalyzer results)
- MatrixCalculator
- MatrixFilter
    - TopKFilter
    - ValueFilter
- MatrixAggregator
- MatrixCombiner
- MatrixFormatter

In addition to these general purpose tools you can use the following specifically geared toward spatial analysis
- DistanceCalculator
- MetricsCalculator
    - IoUMetricsCalculator
    - IoSMetricsCalculator
    - DiceMetricsCalculator 
- OverlapCalculator: computes set of overlap metrics based on IoUMetricsCalculator, IoSMetricsCalculator, DiceMetricsCalculator

## Calculate inter-object distances

Uses PyTorch tensors. The config allows setting a `preferred_device` ("cuda", "mps", "cpu", None). This is not a guarantee. The actual device can be assigned at runtime with `device`. If the device is None, it will be auto-discovered considering the config's `preferred_device`.

In this example we're taking the `centroid` columns from the regions in `l_accepted` dataframe. Because we specify `strict=False` we actually receive the axis-resolved values `centroid-z`, `centroid-y`, `centroid-x`.

We pass the array of centroids as first and second argument to the DistanceCalculator `run` method. This will prompt the pairwise distance calculations between all centroids.

**Note:** We specified `preferred_device=cuda`, however if that device is not found during execution of the `run` method, the calculation will fall back to an alternative device, e.g. MPS (on newer Apple computers) or CPU.

The return type is `MatrixData` which contains the tensor array and optional annotations for each tensor axis. The separation of array and annotations allows keeping the tensors on the device for addiitonal optional downstream matrix operations. It also provides an abstraction that avoids coupling the computational framework to Pandas dataframes which are designed for single-threaded CPU calculations only.

In [ ]:
dccfg = DistanceCalculatorConfig(
    preferred_device="cuda",
)

centroids = dataframe_to_numpy(l_accepted, attributes=["centroid"], strict=False, axes=l_metadata["axes"])
object_ids = dataframe_to_numpy(l_accepted, attributes=["object_id"])
dist = DistanceCalculator(dccfg).run(
    centroids,
    centroids,
    spacing=l_metadata.get("scale", None),
    point_annotations=(object_ids, object_ids),
    device=None,
)

In [ ]:
dist

## Apply a rank filter (k-nearest)

`TopKFilter` selects the *k* smallest or largest values along a chosen axis of a matrix. Like `DistanceCalculator`, matrix filters operate on PyTorch tensors and accept `MatrixData` input directly. When the input carries axis annotations, the filtered result keeps them.

In this example we apply the filter to the pairwise distance matrix `dist` from the previous step. With `k=1`, `axis=1`, and `largest=False`, each **row** keeps only its single nearest neighbor (smallest distance in that row).

`triangle=OFF_DIAGONAL` excludes the diagonal (self-to-self distances). Without this mask, each object would match itself at distance zero.

`output="masked_values"` returns a matrix the same shape as the input: selected entries keep their value and all other cells are set to `NaN`. This preserves shape and axis labels for downstream matrix filters or aggregators.

**Notes:** 
- Other `output` modes are `"indices"`, `"mask"`, and `"values"`. With `mask` or `masked_values` the output is shaped like the input matrix. `mask` returns a boolean array where `True` values indicate array elements that satisfy the filter. `masked_values` returns an array with values where elements that do not fit the filter are masked as `NaN`. Output `indices` and `values` returns 1D arrays of the element indices (or values) that satisfy the filter.
- `axis=0` is column-wise; `axis=1` is row-wise; `axis=None` applies the selection globally.

In [ ]:
tkcfg = TopKFilterConfig(
    k=1,
    axis=1,
    largest=False,
    triangle=OFF_DIAGONAL,
    output="masked_values",
)

tk = TopKFilter(tkcfg).run(dist)
tk

## Apply a threshold filter

`ValueFilter` keeps matrix entries that satisfy a comparison against a reference value (`>`, `<`, `>=`, `<=`, `==`, `!=`). It shares the same `triangle` masking and `output` options as `TopKFilter`.

In this example we threshold the same distance matrix `dist` **column-wise** (`axis=0`): keep entries where the distance is greater than `80.0`. Units follow whatever spacing was passed into `DistanceCalculator.run` — here `l_metadata["scale"]`.

`triangle=LOWER_ND` restricts selection to positions where the row index is greater than the column index (`i > j`), excluding the diagonal. That avoids counting redundant symmetric pairs in an all-pairs distance matrix.

The result `mint` is a `MatrixData` object with the same row/column annotations as `dist`, so downstream steps can still refer to `object_id` axis labels.

**Note:** This step is independent of the `TopKFilter` result above — both filters are applied directly to `dist` to demonstrate different selection modes.

In [ ]:
mincfg = ValueFilterConfig(
    ref_value=80.0,
    axis=0,
    operator=">",
    triangle=LOWER_ND,
    output="masked_values",
)

mint = ValueFilter(mincfg).run(dist)
mint

## Aggregate along an axis

`MatrixAggregator` reduces a matrix along one axis using a built-in operation (`count`, `sum`, `mean`, `min`, `max`, etc.). It accepts torch tensors, ndarrays, boolean masks, or `MatrixData`.

In this example we `count` along `axis=1` on `mint`, the thresholded distance matrix from the previous step. Each row's count is the number of column neighbors that satisfied the distance cutoff for that row object (non-`NaN` entries after `output="masked_values"`).

The return value is a 1-D result indexed by the row annotations from the input — here, `object_id` values from `mint`, which passed through from the upstream input `l_accepted`.

**Note:** With `triangle=LOWER_ND` on the threshold filter, each row can have at most one surviving off-diagonal entry in this toy layout, so counts are typically `0` or `1`.

In [ ]:
macfg = MatrixAggregatorConfig(
    operation="count",
    axis=1,
)

counts = MatrixAggregator(macfg).run(mint)
counts

## MatrixFormatter for `MatrixData`

Use `MatrixFormatter` to convert `MatrixData` into display/export-friendly containers without changing the upstream calculator configs.

In the next cell, we format the distance output (`dist`) with:

- `output_type="dataframe"` to get a pandas DataFrame
- default `annotate=True` to preserve row/column labels from `MatrixData.annotations`

Supported outputs are `"dataframe"`, `"np.ndarray"`, and `"torch.Tensor"`.

**Note:** Keep data as `MatrixData` / tensors while chaining filters and aggregators. Format only when you need a table or explicit array conversion. Annotations are dropped if output_type is a numpy array or Torch tensor.

In [ ]:
mfc = MatrixFormatterConfig(
    output_type="dataframe",
    annotate=True,
)

formatted = MatrixFormatter(mfc).run(dist)
formatted

## Overlap calculation for coincidence detection

`OverlapCalculator` measures spatial coincidence between two label volumes, sets of binary masks, or two sets of bounding boxes. Here, we use the labeled stacks (`labels`) and coarse area labels (`areas`). We pass **region maps** built from the filtered region tables so only accepted `object_id` entries are compared, in dataframe row order.

The config below uses `LabelOverlapCalculatorConfig` with IoU, IoS, and Dice metrics and `intersection_calculator=LabelIntersectionCalculatorConfig(mode="auto")`. Spacing from `l_metadata["scale"]` scales areas and intersections to physical units; the ratios themselves are unchanged under uniform scaling.

Reference for presets and return types:

| Preset | Inputs to `.run(a, b, ...)` |
|--------|-----------------------------|
| `LabelOverlapCalculatorConfig` | 2D/3D/ND **integer label volumes** + `region_map` |
| `BoxOverlapCalculatorConfig` | `(N, 2*D)` box arrays or `region_map` with bboxes only |
| `MaskOverlapCalculatorConfig` | `(N, *spatial)` boolean mask stacks |

On the **label path**, overlap is computed directly from label volumes (`label_areas`, `label_intersection_linear` / `label_intersection_sparse`). No full mask stacks are built, so large volumes (e.g. hundreds of objects on `512×512×Z`) stay memory-efficient.

### Region maps (`object_id` vs `label_id`)

After `RegionAnalyzer` and optional `RegionFilter`, build a map per channel from the accepted region table — **not** the table itself as overlap input:

```python
l_rm = region_map_from_dataframe(l_accepted.reset_index())
a_rm = region_map_from_dataframe(a_accepted.reset_index())
```

Each map entry is keyed by globally unique **`object_id`**. The value holds:

- **`label_id`** — integer label in that channel's label volume (required for the label preset)
- **`bbox`** — optional axis-aligned box (discovered from the volume when omitted)

Row/column order in the overlap matrices follows map key order (dataframe row order). Axis labels on each `MatrixData` metric default to those `object_id` keys.

### Return value (`OverlapResult`)

`.run()` always returns an `OverlapResult` — same type for piping into downstream tasks.

| Field | Default | Notes |
|-------|---------|-------|
| `metrics` | populated | dict of `MatrixData`, e.g. `result.metrics["iou"]` |
| `area_a`, `area_b`, `intersection`, `union` | `None` | set `return_components=True` on the config to include |
| `annotations` | when `region_map` passed | row/column labels from map keys |

Use `MatrixFormatter` to export annotated DataFrames or raw arrays:

```python
MatrixFormatter(MatrixFormatterConfig(output_type="dataframe", annotate=True)).run(result.metric("iou"))
```

### Label intersection mode

On `LabelOverlapCalculatorConfig`, set `intersection_calculator=LabelIntersectionCalculatorConfig(mode=...)`:

- **`"auto"`** (default) — pick linear (histogram) or sparse (bbox crops) from volume size and bbox overlap
- **`"linear"`** — fast when objects are dense or moderately overlapping
- **`"sparse"`** — fast when bboxes are small and mostly disjoint

### Metrics

- **IoU** — intersection / union
- **IoS** — intersection / min(area); 1.0 means one object fully contains the other
- **Dice** — 2×intersection / (area_a + area_b)

In [ ]:
olcfg = LabelOverlapCalculatorConfig(
    return_components=True,  # True to also return area_a, area_b, intersection, union
    intersection_calculator=LabelIntersectionCalculatorConfig(mode="auto"),
    metrics_calculators=metrics_calculator_configs(("iou", "ios", "dice")),
)

# Use all regions in labels and areas stacks.
# reset_index() exposes object_id as a column for region_map_from_dataframe.
l_rm = region_map_from_dataframe(l_regions.reset_index())
a_rm = region_map_from_dataframe(a_regions.reset_index())

calc = OverlapCalculator(olcfg)

# Pass label volumes (not region property tables) as a and b.
overlaps = calc.run(
    labels,
    areas,
    region_map=(l_rm, a_rm),
    spacing=l_metadata.get("scale"),
)

## Overlap Results

`OverlapCalculator.run(...)` returns an `OverlapResult` with one canonical container:

- `overlaps.matrices: dict[str, MatrixData]`

This dict always contains metric keys (for example `"iou"`, `"ios"`, `"dice"`). If `return_components=True`, it also includes geometry keys:

- `"area_a"`, `"area_b"`, `"intersection"`, `"union"`

Use these access patterns:

- `overlaps.metrics` → metrics-only view (excludes component keys)
- `overlaps.metric(name)` → one metric `MatrixData`
  - if there is exactly one metric, `name` can be omitted
  - if multiple metrics exist, `name` is required
- `overlaps.matrices["area_a"]`, `overlaps.matrices["intersection"]`, etc. → component matrices when present
- `overlaps.annotations` → shared row/column annotations inferred from 2-D matrices

In this example, `overlaps.metrics["ios"]` is intersection-over-smaller: `1.0` means one region fully contains the other; `0.0` means no overlap.

In [ ]:
overlaps.annotations  # row/column labels from region_map keys

In [ ]:
overlaps.metrics["ios"]  # 1.0: one object fully contains the other; 0.0: no overlap

In [ ]:
overlaps.metrics["iou"]

## Export annotated DataFrames

Formatting is handled by `MatrixFormatter`, not by the overlap calculator config. Pass any metric `MatrixData` from `OverlapResult` together with a `MatrixFormatterConfig` that sets `output_type` and `annotate`.

In this example we export IoU as a labeled pandas DataFrame. Row and column labels come from the annotations already stored on the metric matrix (the `object_id` keys from `region_map`).

**Note:** Keep metrics as `MatrixData` / tensors when piping into matrix filters or `GraphBuilder`. Use `MatrixFormatter` only when you need a CPU-side table for inspection or export.

In [ ]:
formatter = MatrixFormatter(MatrixFormatterConfig(output_type="dataframe", annotate=True))
iou_df = formatter.run(overlaps.metric("iou"))
iou_df

---

# Build a Containment Graph

We can use the ios metric from the OverlapResult to define a hierarchical graph and region annotations. Overlap metrics and region annotations are linked throughn `object_id`. 

In [ ]:
all_accepted = pd.concat([l_regions, a_regions], axis=0).reset_index().set_index("object_id")

In [ ]:
all_accepted

## Filter IoS matrix

A label is considered to be contained within another label if the IoS>0.5.

In [ ]:
vcfg = ValueFilterConfig(
    ref_value=0.5,
    operator=">",
    output="masked_values",
)
ios_filtered = ValueFilter(vcfg).run(overlaps.metric("ios"))
ios_filtered

In [ ]:
gbc = GraphBuilderConfig(edge_attribute="ios")

G = GraphBuilder(gbc).run(ios_filtered, all_accepted)
G

## GraphLike

`GraphBuilder.run` returns a `GraphLike` graph. The interface mirrors networkx, but node and edge
attributes are read through `node_attrs` / `edge_attrs` instead of `graph.nodes[id]`. By subclassing GraphLike you can switch the default backend NXGraph to your favorite Graph library.

## GraphQuery

Request summary keys via `GraphQueryConfig.attributes`. Defaults cover counts, roots/leaves,
parent/child maps, and edge records.

In [ ]:
from vistiq.graph import GraphQuery, GraphQueryConfig

gq = GraphQuery(
    GraphQueryConfig(
        attributes=[
            "n_nodes",
            "n_edges",
            "roots",
            "leaves",
            "nodes_by_attribute",
            "edges",
        ],
        group_attribute="channel", # works in conjunction with nodes_by_attribute; get count for nodes grouped by channel identifier
        weight_attribute="ios",
    )
)
summary = gq.run(G)
{k:v for k,v in summary.items()}


In [ ]:
pd.DataFrame(summary["edges"])#.head()


In [ ]:
# all_accepted is indexed by object_id, so use the index directly.
node_id = all_accepted.index[0]
print (f"Picking node id {node_id} and find its edges")

gqc = GraphQueryConfig(
    attributes=["filtered_edges", "edges"],
    source_nodes=None # source_node=[node_id],
)

partner_edges = GraphQuery(gqc).run(G)["filtered_edges"]
pd.DataFrame(partner_edges).head()


## GraphFilter

Select node ids or edge records by id list or attribute match. Modes: `nodes`, `edges`,
`direct_path`, `full_path`.

In [ ]:
largest_id = l_regions["volume"].idxmax()
print (f"Region with largest volume: {largest_id}")
matches = GraphFilter(
    GraphFilterConfig(mode="nodes", node_match=[largest_id])
).run(G)
print (f"matched nodes: {matches}")

In [ ]:
from vistiq.graph import (
    GraphFilter,
    GraphFilterConfig,
    GraphQuery,
    GraphQueryConfig,
    GraphQueryFormatter,
    GraphQueryFormatterConfig,
    graph_to_dataframe,
)
# 1) GraphFilter: choose seed nodes (downstream objects of interest)
paths = GraphFilter(
    GraphFilterConfig(
        mode="full_path",
        node_match={
            "target": {"channel": "Area"},
            "source": {"channel": "Label"},
        },
    )
).run(G)
paths_df = pd.DataFrame(paths)
nodes_df = graph_to_dataframe(G).loc[
    sorted({n for e in paths for n in (e["source"], e["target"])})
]
paths_df

In [ ]:
nodes_df

In [ ]:
from pyvis.network import Network

net = Network(notebook=True, cdn_resources='remote', bgcolor="#222222", font_color="white", select_menu=True)
net.barnes_hut()

# Convert the networkx object
net.from_nx(G.raw)
neighbor_map = net.get_adj_list()

# add neighbor data to node hover data
keys = ["object_id", "volume", "aspect_ratio"]
for node in net.nodes:
    vals = "\n".join([f"{k}:{node[k]}" for k in keys])
    node["title"] = f'{node["channel"]}:{node["label"]}\n{vals}'
    node["value"] = len(neighbor_map[node["id"]])

# Render
net.show("nx_graph.html")

In [ ]:
net.show_buttons(filter_=['physics'])

# l